# Model Architecture Explorer
Instantiates every model in the project and prints its architecture + parameter count.
No checkpoint loading — all models use default/random weights.

In [ ]:
import sys; sys.path.insert(0, "../src")
import torch
import torch.nn as nn
import torch.nn.functional as F
import zuko

from suplat.models.byol_models import (
    BYOLEncoder, ProjectionHead, PredictionHead,
    create_efficientnet_b0_backbone,
    BYOLEfficient, BYOLEfficientNetB0,
    PCAProjection,
)
from suplat.models.generative_models import FlowMatchingUNet
from suplat.models.baseline_models import CNN, ScatterNet, SimpleScatterNet, DualScatterSqueezeNet

def param_count(m):
    return f"{sum(p.numel() for p in m.parameters()):,}"

def online_param_count(m):
    """Count params in the online branch (encoder + projector + predictor)."""
    parts = [m.online_encoder, m.online_projector]
    if m.online_predictor is not None:
        parts.append(m.online_predictor)
    total = sum(p.numel() for part in parts for p in part.parameters())
    return f"{total:,}"

# ---------------------------------------------------------------------------
# BYOLFineTuner — inlined from scripts/train_finetuning.py
# (that script calls parse_args() at module level so it cannot be imported)
# ---------------------------------------------------------------------------
class BYOLFineTuner(nn.Module):
    """
    Neural finetuning head: wraps a BYOL online encoder + projector and adds
    a linear classification head on top.
    Note: classifier is hardcoded to 128-dim input (matching a specific training run).
    """
    def __init__(self, byol_model, num_classes=21, training_mode=3, dropout_rate=0.0):
        super().__init__()
        self.encoder    = byol_model.online_encoder
        self.projector  = byol_model.online_projector
        self.training_mode = training_mode
        self.dropout    = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(128, num_classes)  # hardcoded; see script for context

    def forward(self, x):
        z = self.encoder(x)
        z = self.projector(z)
        z = self.dropout(z)
        return self.classifier(z)

# ---------------------------------------------------------------------------
# ProjectionDecoder — inlined from scripts/train_generative.py
# ---------------------------------------------------------------------------
class ProjectionDecoder(nn.Module):
    """FC decoder: z → 89×89. Fast baseline, tends to produce blurry outputs."""
    def __init__(self, proj_dim=256, dropout=0.0):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(proj_dim, 1024), nn.BatchNorm1d(1024), nn.ReLU(inplace=True), nn.Dropout(p=dropout),
            nn.Linear(1024, 256 * 5 * 5), nn.BatchNorm1d(256 * 5 * 5), nn.ReLU(inplace=True), nn.Dropout(p=dropout),
        )
        def up(ic, oc):
            return nn.Sequential(
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                nn.Conv2d(ic, oc, 3, padding=1), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
            )
        self.up1 = up(256, 128); self.up2 = up(128, 64)
        self.up3 = up(64,  32);  self.up4 = up(32,  16)
        self.out_conv = nn.Conv2d(16, 1, 3, padding=1)

    def forward(self, z):
        x = self.fc(z).view(-1, 256, 5, 5)
        x = self.up1(x); x = self.up2(x); x = self.up3(x); x = self.up4(x)
        x = F.interpolate(x, size=(89, 89), mode="bilinear", align_corners=False)
        return torch.sigmoid(self.out_conv(x))

print("Imports OK")

## 1. BYOL Encoder
Custom ResNet-style encoder for 89×89 greyscale images. Outputs a 512-dim representation.

In [ ]:
enc = BYOLEncoder()
print(enc)
print(f"\nParams: {param_count(enc)}")

## 2. BYOL Projection & Prediction Heads
Both are 2-layer MLPs with BN. Projector maps 512 → 256; predictor maps 256 → 256.

In [ ]:
proj = ProjectionHead(in_dim=512, hidden_dim=4096, out_dim=256)
pred = PredictionHead(in_dim=256, hidden_dim=4096, out_dim=256)

print("--- ProjectionHead ---")
print(proj)
print(f"Params: {param_count(proj)}")

print("\n--- PredictionHead ---")
print(pred)
print(f"Params: {param_count(pred)}")

## 3. EfficientNet-B0 Backbone
Pretrained on ImageNet; first conv replaced to accept 1-channel input. Classifier head removed → 1280-dim output.

**Note:** instantiation downloads ImageNet weights (~20 MB) on first run.

In [ ]:
backbone = create_efficientnet_b0_backbone(num_channels=1, dropout_rate=0.2)
print(backbone)
print(f"\nParams: {param_count(backbone)}")

## 4. Full BYOL Models: BYOLEfficient & BYOLEfficientNetB0
Both use `feature_compression_mode='mlp'` so the online predictor is constructed at init time.
The target network is a frozen EMA copy of the online branch and is excluded from the online param count.

In [ ]:
byol_eff = BYOLEfficient(feature_compression_mode='mlp')

print("=== BYOLEfficient ===")
print("--- online_encoder ---"); print(byol_eff.online_encoder)
print("--- online_projector ---"); print(byol_eff.online_projector)
print("--- online_predictor ---"); print(byol_eff.online_predictor)
print(f"\nOnline-branch params: {online_param_count(byol_eff)}")
print(f"Total params (incl. frozen target): {param_count(byol_eff)}")

In [ ]:
byol_effnet = BYOLEfficientNetB0(feature_compression_mode='mlp')

print("=== BYOLEfficientNetB0 ===")
print("--- online_encoder ---"); print(byol_effnet.online_encoder)
print("--- online_projector ---"); print(byol_effnet.online_projector)
print("--- online_predictor ---"); print(byol_effnet.online_predictor)
print(f"\nOnline-branch params: {online_param_count(byol_effnet)}")
print(f"Total params (incl. frozen target): {param_count(byol_effnet)}")

## 5. BYOLFineTuner (neural finetuning head)
Wraps the online encoder + projector from a pretrained `BYOLEfficient` and adds a linear classification head.
The classifier input dim is hardcoded to 128 in the script (matching a specific training run's projection dim).

In [ ]:
# Use a fresh BYOLEfficient with MLP compression as the dummy backbone.
# In practice this is loaded from a checkpoint.
_dummy_byol = BYOLEfficient(projection_dim=128, feature_compression_mode='mlp')
finetuner = BYOLFineTuner(_dummy_byol, num_classes=5)

print("--- classifier head ---")
print(finetuner.classifier)
print(f"Classifier params: {param_count(finetuner.classifier)}")

print("\n--- full finetuner ---")
print(finetuner)
print(f"\nTotal params (encoder + projector + classifier): {param_count(finetuner)}")

## 6. Logistic Regression on BYOL Features
The primary classifier used in evaluation. No neural architecture to print.

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

lr = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', multi_class='ovr')
print(lr)
print()
print("Usage:")
print("  features = byol_model.online_encoder(images).detach().numpy()  # (N, 512)")
print("  projections = byol_model.online_projector(features).numpy()     # (N, proj_dim)")
print("  lr.fit(projections[train_idx], labels[train_idx])")
print("  preds = lr.predict(projections[test_idx])")
print()
print("No learnable parameters — scikit-learn fits coefficients analytically/iteratively.")

## 7. Flow-Matching U-Net Decoder
Velocity network `v_θ(x_t, t, z)` conditioned on BYOL projection `z` and scalar timestep `t`.
U-Net with sinusoidal time embedding + AdaIN-style residual blocks.

In [ ]:
unet = FlowMatchingUNet(z_dim=256, t_dim=128, base_ch=32)
print(unet)
print(f"\nParams: {param_count(unet)}")

## 8. Zuko Neural Spline Flow (NSF)
Conditional normalising flow used to model the distribution of BYOL projections given class labels.
8 rational-quadratic spline transforms with random permutation between layers.

In [ ]:
nsf = zuko.flows.NSF(features=256, context=10, transforms=8, randperm=True)
print(nsf)
print(f"\nParams: {param_count(nsf)}")

## 10. Baseline Classifiers
Classifiers that operate directly on raw images or scattering coefficients.
For 89×89 images with J=2, L=8 the scattering coefficient shape is (81, 23, 23).

In [ ]:
IMG_SHAPE  = (1, 89, 89)
SCAT_SHAPE = (81, 23, 23)  # J=2, L=8 on 89×89 images
N_CLASSES  = 5

cnn = CNN(input_shape=IMG_SHAPE, num_classes=N_CLASSES)
print("--- CNN ---")
print(cnn)
print(f"Params: {param_count(cnn)}")

In [ ]:
scat = ScatterNet(scat_shape=SCAT_SHAPE, num_classes=N_CLASSES, J=2)
print("--- ScatterNet ---")
print(scat)
print(f"Params: {param_count(scat)}")

In [ ]:
simple_scat = SimpleScatterNet(input_shape=SCAT_SHAPE, num_classes=N_CLASSES)
print("--- SimpleScatterNet ---")
print(simple_scat)
print(f"Params: {param_count(simple_scat)}")
print(f"(flat_dim = 81 x 23 x 23 = {81*23*23:,})")

In [ ]:
dual = DualScatterSqueezeNet(img_shape=IMG_SHAPE, scat_shape=SCAT_SHAPE, num_classes=N_CLASSES)
print("--- DualScatterSqueezeNet ---")
print(dual)
print(f"Params: {param_count(dual)}")

## 11. ProjectionDecoder (FC generative baseline)
Fully-connected decoder mapping a BYOL projection `z` back to an 89×89 image via 4 bilinear upsample blocks.
Faster to train than FlowMatchingUNet but produces blurrier outputs.

In [ ]:
dec = ProjectionDecoder(proj_dim=256)
print(dec)
print(f"\nParams: {param_count(dec)}")